# 03 - Crop Province Performance (Gold)

One row per province, crop, year, quarter.
This is the dashboard grain. It keeps the risk label.

In [0]:
from pyspark.sql import functions as F

from configs.project_config import SILVER_TABLES, GOLD_TABLES

source = SILVER_TABLES['crop_production_enriched']
target = GOLD_TABLES['crop_province_performance']

df = spark.table(source)

performance = df.select(
    F.col('canonical_province').alias('province'),
    'crop',
    'crop_group',
    'year',
    'quarter',
    'production',
    F.col('growth_rate_yoy').alias('production_yoy'),
    'production_lag_1',
    'production_lag_4',
    'climate_stress',
    'risk_label'
)

(
    performance.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(target)
)

out = spark.table(target)
print(f'Wrote: {target}')
print(f'Rows : {out.count()}')
print(f'Cols : {out.columns}')
